# Research Notebook — Computational Finance
## Signal Backtesting and Parameter Optimization

This notebook provides the empirical research behind the trading strategy
implemented in the assessment notebook. It explores multiple signals, stocks,
and parameter combinations to justify the final design choices.

### Research Questions
- Which trading signals perform best across different stocks and market conditions?
- What are the optimal parameters for each signal?
- Do the signals remain robust on unseen out-of-sample data?
- How do the signals behave across different sectors and market regimes?
- Does the choice of data period affect signal performance and parameter selection?

### Signals Explored
- Moving Average Crossover (MA)
- Relative Strength Index (RSI)
- Bollinger Bands Mean Reversion
- Moving Average Convergence Divergence (MACD)
- Z-Score Mean Reversion

### Stocks and Indices Explored
- **Financials:** JPM (JPMorgan Chase)
- **Energy:** XOM (ExxonMobil)
- **Healthcare:** JNJ (Johnson & Johnson)
- **Technology:** AAPL (Apple), MSFT (Microsoft)
- **Benchmark:** ^GSPC (S&P 500)

### Data Periods
- **Period 1:** 2010-2025 - avoids 2008 financial crisis structural break
- **Period 2:** 2000-2025 - includes dot-com crash and 2008 crisis for robustness

### In-Sample / Out-of-Sample Split
- **In-sample (parameter optimization):** 2010-2019
- **Out-of-sample (validation):** 2020-2025
- Out-of-sample period deliberately includes COVID crash, rate hike cycle,
  Trump election and tariff war to test signal robustness across extreme
  market regimes

In [11]:
import importlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

import module
importlib.reload(module)

<module 'module' from 'C:\\Users\\User\\PycharmProjects\\JupyterProject\\Computational-Finance\\module.py'>

In [12]:
# Verify all our new functions loaded correctly
print(hasattr(module, 'macd_signal'))
print(hasattr(module, 'zscore_signal'))
print(hasattr(module, 'exponential_moving_average'))
print(hasattr(module, 'rsi_signal'))
print(hasattr(module, 'grid_search_parameters'))
print(hasattr(module, 'split_in_sample_out_of_sample'))

True
True
True
True
True
True


## 1. Data Loading

We load adjusted close prices for all stocks under consideration.
The full period 2010-2025 is then split into in-sample (2010-2019)
and out-of-sample (2020-2025) partitions using the module's built-in
split function.

In [16]:
# Define all stocks to be explored in this research notebook
# Grouped by sector to ensure diversification across signal testing
tickers = [
    'JPM',   # Financials - JPMorgan Chase
    'XOM',   # Energy - ExxonMobil
    'JNJ',   # Healthcare - Johnson & Johnson
    'AAPL',  # Technology - Apple
    'MSFT',  # Technology - Microsoft
    '^GSPC'  # S&P 500 benchmark
]

# Full data period — starts 2010 to avoid 2008 crisis structural break
start_date = '2010-01-01'
end_date   = '2025-12-31'

# Download price data
df_prices, df_price_changes = module.download_stock_price_data(
    tickers, start_date, end_date)

# Split into in-sample and out-of-sample periods
# In-sample  (2010-2019): used for parameter optimization only
# Out-of-sample (2020-2025): used for validation, never touched during optimization
df_is, df_oos = module.split_in_sample_out_of_sample(df_prices, '2019-12-31')

print(f'Full period:       {str(df_prices.index[0])[:10]} to {str(df_prices.index[-1])[:10]} ({len(df_prices)} days)')
print(f'In-sample:         {str(df_is.index[0])[:10]} to {str(df_is.index[-1])[:10]} ({len(df_is)} days)')
print(f'Out-of-sample:     {str(df_oos.index[0])[:10]} to {str(df_oos.index[-1])[:10]} ({len(df_oos)} days)')
print(f'\nStocks loaded: {list(df_prices.columns)}')
df_prices.tail(3)

Full period:       2010-01-04 to 2025-12-30 (4023 days)
In-sample:         2010-01-04 to 2019-12-31 (2516 days)
Out-of-sample:     2020-01-02 to 2025-12-30 (1507 days)

Stocks loaded: ['JPM', 'XOM', 'JNJ', 'AAPL', 'MSFT', '^GSPC']


symbol,JPM,XOM,JNJ,AAPL,MSFT,^GSPC
date,,,,,,
2025-12-26,324.775421,117.523659,206.532059,272.892975,485.547699,6929.939941
2025-12-29,320.655182,118.924744,206.462418,273.252350,484.940430,6905.740234
2025-12-30,320.328369,119.378616,205.815857,272.573578,485.318726,6896.240234


## 2. Signal Exploration (In-Sample: 2010-2019)

We test all four signals on all five stocks using default parameters
on the in-sample period only. This gives us a first overview of which
signals work best on which stocks before we do any parameter optimization.

In [17]:
# Test all signals on all stocks using default parameters (in-sample only)
# We compute the Sharpe ratio for each signal/stock combination
# and display results as a heatmap table

# Stocks to test (excluding benchmark ^GSPC)
test_stocks  = ['JPM', 'XOM', 'JNJ', 'AAPL', 'MSFT']

# Signals to test with their default parameters
signals = {
    'MA Crossover' : lambda s: module.ma_signal(s, 50, 200),
    'RSI'          : lambda s: module.rsi_signal(s),
    'Bollinger'    : lambda s: module.bollinger_signal(s),
    'MACD'         : lambda s: module.macd_signal(s),
    'Z-Score'      : lambda s: module.zscore_signal(s),
}

# Compute Sharpe ratio for each signal/stock combination
# using in-sample data only
prices_is = df_is[test_stocks].copy()
prices_is.index = pd.to_datetime(prices_is.index)

results = {}
for signal_name, signal_fn in signals.items():
    results[signal_name] = {}
    for stock in test_stocks:
        try:
            # Generate signal on in-sample data
            sig_df      = signal_fn(prices_is[stock])
            signal_arr  = sig_df['signal'].to_numpy()

            # Compute daily price returns
            prices_arr  = prices_is[stock].to_numpy()
            daily_ret   = np.concatenate(([0.0],
                          prices_arr[1:] / prices_arr[:-1] - 1))

            # Strategy returns = daily returns * signal (only earn when invested)
            strat_ret   = (daily_ret * signal_arr)[1:]

            # Compute Sharpe ratio
            sharpe      = module.compute_sharpe(strat_ret)
            results[signal_name][stock] = round(sharpe, 3)
        except Exception as e:
            results[signal_name][stock] = np.nan

# Display as a table
df_results = pd.DataFrame(results).T
print('Sharpe Ratios - In-Sample (2010-2019) - Default Parameters')
print('=' * 60)
print(df_results.to_string())
print()
print('Best signal per stock:')
for stock in test_stocks:
    best_signal = df_results[stock].idxmax()
    best_sharpe = df_results[stock].max()
    print(f'  {stock}: {best_signal} (Sharpe: {best_sharpe})')

Sharpe Ratios - In-Sample (2010-2019) - Default Parameters
                JPM    XOM    JNJ   AAPL   MSFT
MA Crossover  0.596 -0.041  0.573  0.969  0.809
RSI           0.450  0.135  0.171  0.121  0.353
Bollinger    -1.058 -1.292 -0.966 -0.950 -0.827
MACD          2.168  1.684  2.268  2.585  2.112
Z-Score      -1.058 -1.292 -0.966 -0.950 -0.827

Best signal per stock:
  JPM: MACD (Sharpe: 2.168)
  XOM: MACD (Sharpe: 1.684)
  JNJ: MACD (Sharpe: 2.268)
  AAPL: MACD (Sharpe: 2.585)
  MSFT: MACD (Sharpe: 2.112)


In [18]:
# Check how often each signal is active on JPM
sig_bb = module.bollinger_signal(prices_is['JPM'])
sig_zs = module.zscore_signal(prices_is['JPM'])

print(f'Bollinger active days: {sig_bb["signal"].sum():.0f} out of {len(sig_bb)}')
print(f'Z-Score active days:   {sig_zs["signal"].sum():.0f} out of {len(sig_zs)}')
print(f'Bollinger trades: {(sig_bb["position_change"] > 0).sum()}')
print(f'Z-Score trades:   {(sig_zs["position_change"] > 0).sum()}')

Bollinger active days: 545 out of 2516
Z-Score active days:   545 out of 2516
Bollinger trades: 47
Z-Score trades:   47


Bollinger Bands and Z-Score produce identical signals on a single asset — a z-score threshold of −2 is just the lower Bollinger Band at ±2σ, which the active-day and trade counts above confirm. Swapping one for the other changes nothing.

The real issue is that all three signals (RSI, Bollinger, Z-Score on one stock) are some form of single-asset mean reversion. The DSR result makes this concrete: Bollinger Bands on MSFT scored 0% — "Likely Luck". 

The fix is to apply Z-Score **cross-sectionally** — comparing each stock's return against its sector peers rather than against its own history. That makes it a relative value signal, which is genuinely different from RSI and MA Crossover.

---
## 3. Signal-to-Industry Assignment

Instead of one stock per signal, each signal is applied to a basket of 5 stocks from the same sector. This way performance reflects something about the sector, not just one company's quirks.

**MA Crossover → Energy** (XOM, CVX, COP, EOG, SLB)  
Energy runs on commodity cycles, oil prices trend for years at a time. That's exactly what a slow MA crossover is built for. All five stocks track crude closely enough that the trend is sector-wide.

**RSI → Technology** (AAPL, MSFT, NVDA, GOOGL, META)  
High-beta tech stocks overshoot on both sides, especially around earnings. RSI captures those overbought/oversold extremes well. Five stocks across different sub-sectors so it's not just one product cycle.

**Cross-sectional Z-Score → Healthcare** (JNJ, PFE, MRK, ABBV, LLY)  
Pharma stocks share the same macro risks (FDA, patent cliffs) but get hit by independent events (trial results, approvals). When one stock drops hard relative to peers, it tends to revert once the market prices in the stock-specific risk properly. The signal goes long the underperformer and exits when it catches back up.

The formula:
$$z_{i,t} = \frac{r_{i,t} - \bar{r}_t}{\sigma^{\text{cross}}_t}$$

Buy when $z_{i,t} < -1.5$, exit when $z_{i,t} \geq 0$.

| Signal | Sector | Logic |
|--------|--------|-------|
| MA Crossover | Energy | Trend: commodity cycles |
| RSI | Technology | Mean reversion: earnings overshoots |
| Z-Score (cross-sectional) | Healthcare | Relative value: peer divergence |